In [0]:
from pyspark.sql.functions import col

base_path = "/Volumes/dev/academy/data/"
process_path = f"{base_path}Process"
rejected_path = f"{base_path}Rejected"
processed_path = f"{base_path}Processed"

paths = [
    process_path,
    processed_path,
    rejected_path
]

for path in paths:
    dbutils.fs.mkdirs(path)
    print(f"Directorio creado: {path}")

print("\nEstructura de carpetas creada.")

In [0]:
from pyspark.sql.functions import col
df = spark.table("samples.tpcds_sf1.inventory")

display(df.limit(5))

In [0]:
r1 = col("inv_quantity_on_hand") < 500
r2 = col("inv_quantity_on_hand") > 100

rules_set = r1 & r2

df_good_rows = df.filter(rules_set)
df_bad_rows = df.filter(~rules_set)

total_count = df.count()
good_count = df_good_rows.count()
bad_count = df_bad_rows.count()

print(f"\nResultados")
print(f"Total de filas leidas: {total_count}")
print(f"Filas Buenas (aprobadas): {good_count}")
print(f"Filas Malas (rechazadas): {bad_count}")

if bad_count > 0:
    print(f"\nGuardando {bad_count} filas malas en formato Parquet en: {rejected_path}")
    df_bad_rows.write.format("parquet").mode("overwrite").save(rejected_path)
else:
    print("\nNo se encontraron filas malas.")

if good_count > 0:
    print(f"Guardando {good_count} filas buenas en formato Parquet en: {processed_path}")
    df_good_rows.write.format("parquet").mode("overwrite").save(processed_path)
else:
    print("No se encontraron filas buenas para procesar.")